# Эксперименты и сравнение моделей

Здесь представлено сравнение моделей сначала на очищенных данных без feature engineering, затем на данных с добавленными признаками. После этого для моделей с FE подбираются гиперпараметры с помощью GridSearchCV и RandomizedSearchCV.

In [2]:
import sys
from pathlib import Path

import pandas as pd
from joblib import dump
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, train_test_split as sklearn_train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from tqdm.auto import tqdm

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT))

from src.modeling import build_model_pipeline, evaluate_classifier, load_raw_data, split_data
from src.preprocessing import TARGET_COLUMN, clean_data, prepare_data

RANDOM_STATE = 42
MODEL_PATH = PROJECT_ROOT / 'models' / 'best_model_cp2.joblib'
METRICS_PATH = PROJECT_ROOT / 'data' / 'processed' / 'metrics_cp2.csv'
TUNING_RESULTS_PATH = PROJECT_ROOT / 'data' / 'processed' / 'tuning_results_cp2.csv'
TEST_METRICS_PATH = PROJECT_ROOT / 'data' / 'processed' / 'test_metrics_cp2.csv'


c:\Users\user\Desktop\ml-project\hseml-group-project-ruf019\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
raw_df = load_raw_data()
clean_df = clean_data(raw_df)
fe_df = prepare_data(raw_df)

clean_train, clean_val, clean_test = split_data(clean_df)
fe_train, fe_val, fe_test = split_data(fe_df)

pd.DataFrame(
    [
        {'dataset': 'raw_df', 'rows': raw_df.shape[0], 'cols': raw_df.shape[1]},
        {'dataset': 'clean_df', 'rows': clean_df.shape[0], 'cols': clean_df.shape[1]},
        {'dataset': 'fe_df', 'rows': fe_df.shape[0], 'cols': fe_df.shape[1]},
    ]
)

,dataset,rows,cols
0,raw_df,119390,32
1,clean_df,86638,30
2,fe_df,86380,38


## 1. Модели на данных без feature engineering

In [4]:
model_specs = [
    ('logistic_regression', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
    ('knn_classifier', KNeighborsClassifier(n_neighbors=15)),
    ('decision_tree', DecisionTreeClassifier(max_depth=8, random_state=RANDOM_STATE)),
    ('random_forest', RandomForestClassifier(n_estimators=120, max_depth=14, n_jobs=1, random_state=RANDOM_STATE)),
    ('gradient_boosting', GradientBoostingClassifier(random_state=RANDOM_STATE)),
]

x_clean_train = clean_train.drop(columns=[TARGET_COLUMN])
y_clean_train = clean_train[TARGET_COLUMN]
x_clean_val = clean_val.drop(columns=[TARGET_COLUMN])
y_clean_val = clean_val[TARGET_COLUMN]

clean_rows = []
clean_models = {}

for model_name, estimator in model_specs:
    pipeline = build_model_pipeline(clean_train, estimator)
    pipeline.fit(x_clean_train, y_clean_train)
    metrics = evaluate_classifier(pipeline, x_clean_val, y_clean_val)
    clean_rows.append({'data_version': 'clean_data', 'model': model_name, **metrics})
    clean_models[model_name] = pipeline

In [5]:
clean_results = pd.DataFrame(clean_rows).sort_values('f1', ascending=False).reset_index(drop=True)
clean_results

,data_version,model,accuracy,precision,recall,f1,roc_auc
0,clean_data,gradient_boosting,0.828793,0.754731,0.565314,0.646433,0.889197
1,clean_data,decision_tree,0.810172,0.677217,0.600611,0.636618,0.869268
2,clean_data,knn_classifier,0.800939,0.675582,0.540578,0.600587,0.838760
3,clean_data,logistic_regression,0.796476,0.685625,0.489161,0.570965,0.843838
4,clean_data,random_forest,0.812327,0.817534,0.414675,0.550249,0.886436


## 2. Те же модели на данных с feature engineering

In [6]:
x_fe_train = fe_train.drop(columns=[TARGET_COLUMN])
y_fe_train = fe_train[TARGET_COLUMN]
x_fe_val = fe_val.drop(columns=[TARGET_COLUMN])
y_fe_val = fe_val[TARGET_COLUMN]

fe_rows = []
fe_models = {}

for model_name, estimator in model_specs:
    pipeline = build_model_pipeline(fe_train, estimator)
    pipeline.fit(x_fe_train, y_fe_train)
    metrics = evaluate_classifier(pipeline, x_fe_val, y_fe_val)
    fe_rows.append({'data_version': 'feature_engineering', 'model': model_name, **metrics})
    fe_models[model_name] = pipeline

In [7]:
fe_results = pd.DataFrame(fe_rows).sort_values('f1', ascending=False).reset_index(drop=True)
fe_results

,data_version,model,accuracy,precision,recall,f1,roc_auc
0,feature_engineering,gradient_boosting,0.835379,0.757587,0.589210,0.662873,0.894895
1,feature_engineering,decision_tree,0.815389,0.692384,0.590053,0.637136,0.870959
2,feature_engineering,knn_classifier,0.799028,0.669627,0.529643,0.591465,0.839172
3,feature_engineering,logistic_regression,0.796867,0.684148,0.483844,0.566820,0.852067
4,feature_engineering,random_forest,0.816084,0.821311,0.422310,0.557803,0.889401


## 3. Сводное сравнение

In [8]:
comparison = pd.concat([clean_results, fe_results], ignore_index=True)
comparison = comparison.sort_values(['f1', 'data_version'], ascending=[False, False]).reset_index(drop=True)
comparison

,data_version,model,accuracy,precision,recall,f1,roc_auc
0,feature_engineering,gradient_boosting,0.835379,0.757587,0.589210,0.662873,0.894895
1,clean_data,gradient_boosting,0.828793,0.754731,0.565314,0.646433,0.889197
2,feature_engineering,decision_tree,0.815389,0.692384,0.590053,0.637136,0.870959
3,clean_data,decision_tree,0.810172,0.677217,0.600611,0.636618,0.869268
4,clean_data,knn_classifier,0.800939,0.675582,0.540578,0.600587,0.838760
5,feature_engineering,knn_classifier,0.799028,0.669627,0.529643,0.591465,0.839172
6,clean_data,logistic_regression,0.796476,0.685625,0.489161,0.570965,0.843838
7,feature_engineering,logistic_regression,0.796867,0.684148,0.483844,0.566820,0.852067
8,feature_engineering,random_forest,0.816084,0.821311,0.422310,0.557803,0.889401
9,clean_data,random_forest,0.812327,0.817534,0.414675,0.550249,0.886436


In [9]:
comparison.pivot(index='model', columns='data_version', values='f1').sort_values('feature_engineering', ascending=False)

data_version,clean_data,feature_engineering
model,,
gradient_boosting,0.646433,0.662873
decision_tree,0.636618,0.637136
knn_classifier,0.600587,0.591465
logistic_regression,0.570965,0.566820
random_forest,0.550249,0.557803


feature engineering в целом оказался полезен, но не для всех моделей одинаково. Лучший результат по F1-score показал gradient_boosting, и после добавления новых признаков его качество выросло с 0.6464 до 0.6629.

Для decision_tree и random_forest улучшение почти незаметное: 0.6366 => 0.6371 у decision_tree и 0.5502 => 0.5578 у random_forest. Улучшение, хоть и небольшое, но есть.

У knn_classifier и logistic_regression качество после feature engineering немного снизилось. Разные модели по-разному чувствительны к структуре признакового пространства.

FE лучше всего сработал для бустинга, а для более простых или чувствительных моделей пользы не дал. Еще на этапе теста на простых очищенных данных gradient_boosting дал лучшие результаты, а с добавление FE его метрики увеличились. Несмотря на то, что метрики baseline модели стали хуже, я считаю, что feature engineering пошёл на пользу.

## 4. Подбор гиперпараметров

Так как основной метрикой является F1-score, поэтому для подбора гиперпараметров в GridSearchCV и RandomizedSearchCV используется scoring='f1'. Подбор оценивается через кросс-валидацию cv=3 на train-части, после этого лучшие параметры заново обучаются на полном train и проверяются на validation-выборке.

Для LogisticRegression используется GridSearchCV, потому что у неё сетка параметров небольшая, и можно полностью перебрать все комбинации "C", "class_weight" и "solver".

Для KNN, DecisionTree, RandomForest и GradientBoosting используется RandomizedSearchCV, так как у этих моделей пространство параметров заметно больше, и полный перебор всех параметров занял бы слишком много времени. Случайный поиск позволяет проверить ограниченное число комбинаций и всё равно провести систематический тюнинг.

Подбор гиперпараметров проводится на подвыборке train, чтобы перебор был не слишком долгим. После выбора лучших параметров каждая модель заново обучается на полном train-наборе.

In [10]:
TUNING_SAMPLE_SIZE = 12000

if len(fe_train) > TUNING_SAMPLE_SIZE:
    tuning_train, _ = sklearn_train_test_split(
        fe_train,
        train_size=TUNING_SAMPLE_SIZE,
        random_state=RANDOM_STATE,
        stratify=fe_train[TARGET_COLUMN],
    )
else:
    tuning_train = fe_train.copy()

x_tuning_train = tuning_train.drop(columns=[TARGET_COLUMN])
y_tuning_train = tuning_train[TARGET_COLUMN]

pd.DataFrame(
    {
        'dataset': ['full_train', 'tuning_train'],
        'rows': [len(fe_train), len(tuning_train)],
        'target_rate': [fe_train[TARGET_COLUMN].mean(), tuning_train[TARGET_COLUMN].mean()],
    }
)


,dataset,rows,target_rate
0,full_train,60466,0.274700
1,tuning_train,12000,0.274667


### Сетки параметров

Для RandomizedSearchCV параметр "n_iter" ограничивает количество случайно проверяемых комбинаций.

In [11]:
tuning_specs = [
    {
        'model': 'logistic_regression',
        'search': 'GridSearchCV',
        'estimator': LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
        'params': {
            'model__C': [0.1, 0.5, 1.0, 3.0, 10.0],
            'model__class_weight': [None, 'balanced'],
            'model__solver': ['liblinear'],
        },
    },
    {
        'model': 'knn_classifier',
        'search': 'RandomizedSearchCV',
        'estimator': KNeighborsClassifier(n_jobs=-1),
        'params': {
            'model__n_neighbors': [7, 11, 15, 21, 31],
            'model__weights': ['uniform', 'distance'],
            'model__p': [1, 2],
        },
        'n_iter': 8,
    },
    {
        'model': 'decision_tree',
        'search': 'RandomizedSearchCV',
        'estimator': DecisionTreeClassifier(random_state=RANDOM_STATE),
        'params': {
            'model__max_depth': [6, 8, 10, 14, None],
            'model__min_samples_split': [2, 5, 10],
            'model__min_samples_leaf': [1, 2, 5],
            'model__class_weight': [None, 'balanced'],
        },
        'n_iter': 14,
    },
    {
        'model': 'random_forest',
        'search': 'RandomizedSearchCV',
        'estimator': RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=1),
        'params': {
            'model__n_estimators': [100, 150, 200, 300],
            'model__max_depth': [10, 14, 18, None],
            'model__min_samples_split': [2, 5, 10],
            'model__min_samples_leaf': [1, 2, 4],
            'model__max_features': ['sqrt', 'log2'],
            'model__class_weight': [None, 'balanced'],
        },
        'n_iter': 16,
    },
    {
        'model': 'gradient_boosting',
        'search': 'RandomizedSearchCV',
        'estimator': GradientBoostingClassifier(random_state=RANDOM_STATE),
        'params': {
            'model__n_estimators': [100, 150, 200, 250],
            'model__learning_rate': [0.03, 0.05, 0.1, 0.15],
            'model__max_depth': [2, 3, 4],
            'model__min_samples_leaf': [1, 2, 5],
            'model__subsample': [0.8, 0.9, 1.0],
        },
        'n_iter': 16,
    },
]


### Запуск подбора

Для каждой модели сначала проводится поиск лучших гиперпараметров на tuning_train, затем модель с найденными параметрами переобучается на полном fe_train. В итоговую таблицу сохраняются метрики на validation-выборке.


In [12]:
tuned_rows = []
tuned_models = {}

for spec in tqdm(tuning_specs, desc='Подбор гиперпараметров'):
    search_pipeline = build_model_pipeline(tuning_train, spec['estimator'])

    if spec['search'] == 'GridSearchCV':
        search = GridSearchCV(
            estimator=search_pipeline,
            param_grid=spec['params'],
            scoring='f1',
            cv=3,
            n_jobs=-1,
        )
    else:
        search = RandomizedSearchCV(
            estimator=search_pipeline,
            param_distributions=spec['params'],
            n_iter=spec['n_iter'],
            scoring='f1',
            cv=3,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )

    search.fit(x_tuning_train, y_tuning_train)

    best_pipeline = build_model_pipeline(fe_train, spec['estimator'])
    best_pipeline.set_params(**search.best_params_)
    best_pipeline.fit(x_fe_train, y_fe_train)
    metrics = evaluate_classifier(best_pipeline, x_fe_val, y_fe_val)

    tuned_rows.append(
        {
            'data_version': 'feature_engineering_tuned',
            'model': spec['model'],
            'search': spec['search'],
            'best_cv_f1': search.best_score_,
            'best_params': search.best_params_,
            **metrics,
        }
    )
    tuned_models[spec['model']] = best_pipeline

tuned_results = pd.DataFrame(tuned_rows).sort_values('f1', ascending=False).reset_index(drop=True)


Подбор гиперпараметров: 100%|██████████| 5/5 [04:28<00:00, 53.76s/it]


### Сводная таблица tuned-моделей


In [13]:
tuned_results_display = tuned_results[
    ['model', 'search', 'accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'best_cv_f1']
].sort_values('f1', ascending=False).reset_index(drop=True)

tuned_results_display


,model,search,accuracy,precision,recall,f1,roc_auc,best_cv_f1
0,random_forest,RandomizedSearchCV,0.826657,0.645276,0.819331,0.721961,0.905577,0.695460
1,gradient_boosting,RandomizedSearchCV,0.849657,0.760258,0.661141,0.707244,0.913746,0.676745
2,decision_tree,RandomizedSearchCV,0.762831,0.541298,0.894914,0.674574,0.880891,0.659256
3,logistic_regression,GridSearchCV,0.759435,0.542516,0.792357,0.644056,0.852030,0.638120
4,knn_classifier,RandomizedSearchCV,0.801111,0.664104,0.558303,0.606625,0.836138,0.534835


### Лучшие параметры

Чтобы лучшие параметры не обрезались при выводе в таблицу, я по-отдельности вывел их на каждую строку (одна строка — один параметр конкретной модели).

In [14]:
best_params_table = pd.DataFrame(
    [
        {
            'model': row['model'],
            'parameter': parameter.replace('model__', ''),
            'value': value,
        }
        for _, row in tuned_results.iterrows()
        for parameter, value in row['best_params'].items()
    ]
)

best_params_table.sort_values(['model', 'parameter']).reset_index(drop=True)

,model,parameter,value
0,decision_tree,class_weight,balanced
1,decision_tree,max_depth,10
2,decision_tree,min_samples_leaf,5
3,decision_tree,min_samples_split,5
4,gradient_boosting,learning_rate,0.15
5,gradient_boosting,max_depth,4
6,gradient_boosting,min_samples_leaf,2
7,gradient_boosting,n_estimators,200
8,gradient_boosting,subsample,0.8
9,knn_classifier,n_neighbors,11


In [15]:
fixed_f1 = fe_results.set_index('model')['f1']
fixed_roc_auc = fe_results.set_index('model')['roc_auc']

model_comparison_after_tuning = tuned_results.copy()
model_comparison_after_tuning['fixed_f1'] = model_comparison_after_tuning['model'].map(fixed_f1)
model_comparison_after_tuning['delta_f1'] = model_comparison_after_tuning['f1'] - model_comparison_after_tuning['fixed_f1']
model_comparison_after_tuning['fixed_roc_auc'] = model_comparison_after_tuning['model'].map(fixed_roc_auc)
model_comparison_after_tuning['delta_roc_auc'] = model_comparison_after_tuning['roc_auc'] - model_comparison_after_tuning['fixed_roc_auc']

model_comparison_after_tuning_display = model_comparison_after_tuning[
    [
        'model',
        'search',
        'fixed_f1',
        'f1',
        'delta_f1',
        'fixed_roc_auc',
        'roc_auc',
        'delta_roc_auc',
    ]
].sort_values('f1', ascending=False).reset_index(drop=True)

model_comparison_after_tuning_display


,model,search,fixed_f1,f1,delta_f1,fixed_roc_auc,roc_auc,delta_roc_auc
0,random_forest,RandomizedSearchCV,0.557803,0.721961,0.164158,0.889401,0.905577,0.016176
1,gradient_boosting,RandomizedSearchCV,0.662873,0.707244,0.044370,0.894895,0.913746,0.018851
2,decision_tree,RandomizedSearchCV,0.637136,0.674574,0.037438,0.870959,0.880891,0.009932
3,logistic_regression,GridSearchCV,0.566820,0.644056,0.077236,0.852067,0.852030,-0.000037
4,knn_classifier,RandomizedSearchCV,0.591465,0.606625,0.015160,0.839172,0.836138,-0.003034


В таблице выше сравниваются tuned-модели с теми же алгоритмами на фиксированных параметрах.

Столбец "fixed_f1" показывает качество до тюнинга, "f1" — качество после подбора гиперпараметров, а "delta_f1" — изменение качества.

Если "delta_f1" положительный, то тюнинг улучшил баланс между "precision" и "recall", если около нуля или отрицательный, значит базовые параметры уже были достаточно удачными или модель хуже переносит найденные параметры на validation-выборку.

### Выводы после тюнинга

Лучший результат по основной метрике F1-score показал RandomForestClassifier — F1 = 0.721961. До подбора гиперпараметров эта же модель имела F1 = 0.557803, то есть тюнинг дал прирост +0.164158. Это самый большой прирост среди всех моделей.

Эта модель после тюнинга использует следующие параметры: class_weight=balanced, max_depth=None, max_features='sqrt', min_samples_leaf=4, min_samples_split=5, n_estimators=200.

GradientBoostingClassifier занял второе место по F1-score = 0.707244, но показал лучший ROC-AUC = 0.9137. Это значит, что градиентный бустинг лучше ранжирует бронирования по вероятности отмены, но по основной метрике уступает случайному лесу.

DecisionTreeClassifier показал F1-score = 0.674574, увеличил свой предыдущий результат (на фиксированных параметрах) на 0.037438.

LogisticRegression сильно улучшилась по F1-score с 0.5668 до 0.6441.

KNN показала самый слабый результат среди tuned-моделей: F1 = 0.6066.

Лучшей моделью после подбора гиперпараметров стала модель случайного леса RandomForestClassifier.

## 5. Выбор и сохранение лучшей модели


In [ ]:
best_row = tuned_results.sort_values('f1', ascending=False).iloc[0]

best_row[
    ['model', 'search', 'accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'best_cv_f1']
]

model              random_forest
search        RandomizedSearchCV
accuracy                0.826657
precision               0.645276
recall                  0.819331
f1                      0.721961
roc_auc                 0.905577
best_cv_f1               0.69546
Name: 0, dtype: object

### Финальная оценка на test

До этого, для подбора гиперпараметров и выбора лучшей модели, они сравнивались на validation-выборке. Теперь прогоним нашу лучшую модель (RandomForestClassifier) на test-выборке, что дать ей финальную оценку.

In [ ]:
best_model = tuned_models[best_row['model']]

x_fe_test = fe_test.drop(columns=[TARGET_COLUMN])
y_fe_test = fe_test[TARGET_COLUMN]

test_metrics = evaluate_classifier(best_model, x_fe_test, y_fe_test)
test_results = pd.DataFrame(
    [
        {
            'data_version': 'feature_engineering_test',
            'model': best_row['model'],
            'search': best_row['search'],
            **test_metrics,
        }
    ]
)

test_results

,data_version,model,search,accuracy,precision,recall,f1,roc_auc
0,feature_engineering_test,random_forest,RandomizedSearchCV,0.826812,0.644284,0.824951,0.723509,0.908421


На test-выборке модель показала F1 = 0.7235, это чуть больше к результата на validation-выборке (F1 = 0.7220). Значит модель не переобучилась и качество устойчиво переносится на новые данные.

Также модель показала ROC-AUC = 0.9084, то есть она хорошо ранжирует бронирования по вероятности отмены, recall = 0.8250 показывает, что модель находит большую часть реальных отмен, precision = 0.6443 показывает, что среди предсказанных отмен примерно 64% действительно являются отменами.

In [ ]:
all_metrics = pd.concat([comparison, tuned_results, test_results], ignore_index=True)

MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
dump(best_model, MODEL_PATH)
all_metrics.to_csv(METRICS_PATH, index=False)
tuned_results.to_csv(TUNING_RESULTS_PATH, index=False)
test_results.to_csv(TEST_METRICS_PATH, index=False)

MODEL_PATH, METRICS_PATH, TUNING_RESULTS_PATH, TEST_METRICS_PATH

(WindowsPath('C:/Users/user/Desktop/ml-project/hseml-group-project-ruf019/models/best_model_cp2.joblib'),
 WindowsPath('C:/Users/user/Desktop/ml-project/hseml-group-project-ruf019/data/processed/metrics_cp2.csv'),
 WindowsPath('C:/Users/user/Desktop/ml-project/hseml-group-project-ruf019/data/processed/tuning_results_cp2.csv'),
 WindowsPath('C:/Users/user/Desktop/ml-project/hseml-group-project-ruf019/data/processed/test_metrics_cp2.csv'))